# Segmentation 2: segment all frames (10 degree mosaicity)

Runs the segmentation from `01_find_parameters.ipynb` on every frame of every
translation, one process per translation, and writes all peaks to one ImageD11
columnfile (`processed/domains_mosaicity_10p0deg/peaks_2d.h5`).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# make the repository importable, whatever the working directory is
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simtools").is_dir())
sys.path.insert(0, str(REPO))

import json
import os
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from tqdm.auto import tqdm
from simtools import io, paths, segmenter

In [2]:
SAMPLE = "domains_mosaicity_10p0deg"
with open(paths.segmenter_options_path(SAMPLE)) as f:
    options = json.load(f)
options

{'norm_factor': 45, 'threshold': 0}

In [3]:
files = io.scan_files(SAMPLE)
n_workers = min(len(files), len(os.sched_getaffinity(0)))
print(f"segmenting {len(files)} translations with {n_workers} processes")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=n_workers) as pool:
    results = list(tqdm(pool.map(segmenter.segment_scan, files, [options] * len(files)), total=len(files)))
print(f"done in {time.perf_counter() - t0:.0f} s")

pks = {k: np.concatenate([r[k] for r in results]) for k in results[0]}
print(f"{len(pks['omega'])} peaks in total")

segmenting 103 translations with 32 processes


  0%|          | 0/103 [00:00<?, ?it/s]

done in 128 s
4872053 peaks in total


In [4]:
peaks_path = paths.peaks_path(SAMPLE)
if os.path.exists(peaks_path):
    os.remove(peaks_path)
colf = io.write_peaks(pks, peaks_path)
print("wrote", peaks_path)

wrote /dtu-compute/msaca/adaptive_basis_simulations/processed/domains_mosaicity_10p0deg/peaks_2d.h5
